# Sequential Architecture — LangChain

**Pattern:** Sequential (A → B → C pipeline)  
**Framework:** LangChain LCEL  
**Task:** Research city data → Summarize → Format travel report

## What Is the Sequential Pattern?

```
Input
  │
  ▼
┌─────────────┐
│  Researcher │  ← fetches + structures raw data
└──────┬──────┘
       │ structured_facts
       ▼
┌─────────────┐
│  Summarizer │  ← writes 2-sentence summary
└──────┬──────┘
       │ summary
       ▼
┌─────────────┐
│  Formatter  │  ← formats into report section
└──────┬──────┘
       │ report_section
       ▼
    Output
```

**Key property:** No feedback loops. No branching. Each step's output is the next step's input.

**LangChain implementation:** Three LCEL chains (`prompt | llm | parser`). Data passed as Python variables — no framework magic.

## Setup

In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "Set GOOGLE_API_KEY in .env"

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0)
parser = StrOutputParser()
print("✓ Ready")

## Mock Data Source

In [ ]:
def fetch_city_data(city: str) -> str:
    data = {
        "tokyo":     "Weather: Clear, 18°C. Safety: Low. Time: 22:30 JST. Highlights: Shibuya, Senso-ji.",
        "paris":     "Weather: Partly Cloudy, 16°C. Safety: Low. Time: 15:30 CET. Highlights: Eiffel Tower, Louvre.",
        "bangalore": "Weather: Rainy, 25°C. Safety: Medium. Time: 20:00 IST. Highlights: Lalbagh, Nandi Hills.",
    }
    return data.get(city.lower(), f"No data for {city}.")

print(fetch_city_data("Tokyo"))

## Three LCEL Chains — One per Step

Each chain is `prompt | llm | parser`. Data flows via Python variables — no framework magic needed for sequential pipelines.

In [ ]:
researcher_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a travel data researcher. Extract and structure the key facts."),
        ("human", "City: {city}\nRaw data: {raw_data}\n\nExtract: weather, safety, time, top 2 attractions."),
    ])
    | llm | parser
)

summarizer_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a travel writer. Write a concise 2-sentence city summary."),
        ("human", "Structured facts:\n{structured_facts}\n\nWrite a traveler summary."),
    ])
    | llm | parser
)

formatter_chain = (
    ChatPromptTemplate.from_messages([
        ("system", "You are a report editor. Format into a polished report section with a markdown header."),
        ("human", "City: {city}\nSummary: {summary}\n\nFormat with '### {city}' header."),
    ])
    | llm | parser
)

print("3 chains ready: researcher → summarizer → formatter")

## Run the Sequential Pipeline

In [ ]:
def run_pipeline(city: str) -> str:
    raw_data = fetch_city_data(city)
    structured_facts = researcher_chain.invoke({"city": city, "raw_data": raw_data})
    summary = summarizer_chain.invoke({"structured_facts": structured_facts})
    report_section = formatter_chain.invoke({"city": city, "summary": summary})
    return report_section

# Run for all three cities
cities = ["Tokyo", "Paris", "Bangalore"]
sections = [run_pipeline(city) for city in cities]

print("# Travel Report\n")
for section in sections:
    print(section)
    print()

## Key Takeaways

| What You Saw | Why It Matters |
|---|---|
| 3 LCEL chains chained by Python variables | Sequential = no framework needed, just function composition |
| No shared state, no memory | Each step is stateless — only its inputs matter |
| `prompt \| llm \| parser` | LCEL operator syntax chains components into a Runnable |
| Same mock data as all frameworks | Outputs are directly comparable across LangChain/LangGraph/CrewAI/ADK |

### When to Use Sequential
- Clear A→B→C pipeline with no feedback
- Each step transforms the previous step's output
- No need for shared state or retries

### Next: [02-Parallel →](../../02-Parallel/LangChain/parallel.ipynb)